In [ ]:
import re
import subprocess
from   pathlib import Path

GAME                 = "nine_mens_morris"
GENERATED_CODE_PATH  = f"outputs/{GAME}.py"

LLM_MODEL            = ""
TIMEOUT_SECONDS      = 600
OPEN_SPIEL_MAX_STEPS = 500

CODE_PATH            = Path(GENERATED_CODE_PATH)
RESPONSE_PATH        = CODE_PATH.with_suffix(".md")

## LLM implementation

This optional cell generates code from the local prompt and rule files.


In [ ]:
try:
    if not LLM_MODEL:
        raise ValueError("Set LLM_MODEL")

    prompt_text = Path("input/prompt.txt").read_text(encoding="utf-8")
    rules_text = Path("input/game_rules.txt").read_text(encoding="utf-8")
    full_prompt = prompt_text + "\n\nHier folgt die Spielanleitung:\n\n" + rules_text

    result = subprocess.run(
        ["pi", "-p", "--model", LLM_MODEL, full_prompt],
        capture_output=True,
        text=True,
        timeout=TIMEOUT_SECONDS,
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip() or "pi call failed")

    CODE_PATH.parent.mkdir(parents=True, exist_ok=True)
    RESPONSE_PATH.write_text(result.stdout, encoding="utf-8")

    match = re.search(r"```python\s*(.*?)```", result.stdout, re.IGNORECASE | re.DOTALL)
    if match is None:
        raise RuntimeError("No fenced python block found in the LLM response")

    CODE_PATH.write_text(match.group(1).strip() + "\n", encoding="utf-8")
except Exception as exc:
    print(f"LLM call failed: {exc}")


## OpenSpiel and generated game loading


In [ ]:
import importlib.util
import sys
import pyspiel

try:
    game = pyspiel.load_game(GAME)

    if not CODE_PATH.exists():
        raise FileNotFoundError(f"Generated code missing: {CODE_PATH}")

    spec = importlib.util.spec_from_file_location(GAME, CODE_PATH)
    module = importlib.util.module_from_spec(spec)
    if spec is None or spec.loader is None:
        raise RuntimeError("Could not load generated game module")
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    llm_game = module.Game()
except Exception as exc:
    print(f"Game loading failed: {exc}")


## Random test on both games


In [ ]:
import random

try:
    print("OpenSpiel random rollout (separate; raw OpenSpiel action ids)")
    state = game.new_initial_state()
    steps = 0

    while not state.is_terminal():
        player = state.current_player()
        actions = list(state.legal_actions())
        if not actions:
            raise RuntimeError(f"OpenSpiel has no legal actions before terminal at step {steps}")

        action = random.choice(actions)
        print(f"OpenSpiel step {steps}: player={player} action={action}")
        state.apply_action(action)

        steps += 1
        if steps > OPEN_SPIEL_MAX_STEPS:
            raise RuntimeError(f"Too many OpenSpiel steps: {OPEN_SPIEL_MAX_STEPS}")

    print(f"OpenSpiel terminal after {steps} steps, returns={state.returns()}")

    print("Generated game random rollout (separate; canonical generated action names)")
    llm_state = llm_game.initial_state()
    steps = 0

    while not llm_game.is_terminal(llm_state):
        player = llm_game.current_player(llm_state)
        llm_actions = list(llm_game.legal_actions(llm_state))
        if not llm_actions:
            raise RuntimeError(f"Generated game has no legal actions before terminal at step {steps}")

        action = random.choice(llm_actions)
        action_name = llm_game.action_to_name(action)
        if llm_game.name_to_action(action_name) != action:
            raise RuntimeError(f"Generated action name did not round-trip at step {steps}: {action_name}")

        print(f"Generated step {steps}: player={player} action={action_name}")
        next_state = llm_game.apply_action(llm_state, action)
        if next_state is not None:
            llm_state = next_state

        steps += 1
        if steps > OPEN_SPIEL_MAX_STEPS:
            raise RuntimeError(f"Too many generated-game steps: {OPEN_SPIEL_MAX_STEPS}")

    print(f"Generated game terminal after {steps} steps, returns={llm_game.returns(llm_state)}")
except NameError:
    print("Random test failed: load the games first")
except Exception as exc:
    print(f"Random test failed: {exc}")
